In [ ]:
import pandas as pd
from functions import search_files, clean_ft

In [ ]:
modify_src = pd.read_csv(r"C:\Users\mturky\Downloads\31060067_MODIFYSUBSTYPERPT.CSV",dtype='str')

detailed_src = pd.read_csv(r"C:\Users\mturky\Downloads\31060231_DETAILNEWANDREN.CSV",dtype='str')
filename = search_files('BEINDATANEWRPT')[1]
print(filename)
beindata_src= pd.read_csv(filename, dtype='str')


In [ ]:
bein = beindata_src.copy()

bien work

In [ ]:
beindata_src.loc[(beindata_src['Plan'].str.contains('6 Months',case=False,na=False)) 
                 & (beindata_src['Customer Type'].isin(['beIN Quartar Installment', 'beIN Bi Installment','BeIN sports CC',])) 
                 & (beindata_src['Status']=='Active') ].to_csv('wrong_plan_type.csv', index=False)


modify details 

In [ ]:
in_modify_not_detailed =  modify_src.loc[(~modify_src['Customer Number'].isin(detailed_src['Customer Number']))]

In [ ]:
in_modify_not_detailed = in_modify_not_detailed.loc[(~in_modify_not_detailed['New Customer Type'].isin(['Head End','Illegal Network'])) & (in_modify_not_detailed['USER_ENTITY']!='CNE Head office')]
in_modify_not_detailed.to_csv('in_modify_not_detailed.csv', index= False)

# to be continued after having the "metnayel 3la 3eno futuer disconnection report"

all users history details

In [ ]:
history_src = pd.read_csv(r"C:\Users\mturky\Downloads\31065218_ALLUSERHISTDTLS.CSV", dtype='str')

In [ ]:
history_change_depot = history_src.loc[(history_src['Transaction Type']=='Modify User Operational Entity') & (history_src['Transaction User']!='Super User')]
history_change_depot.to_csv('users wrongly change depots.csv')

wrong entity on action

In [ ]:
ft_src = clean_ft(search_files('NEWCNEFINTRANSRPT')[0])


In [ ]:
bein.columns

In [ ]:
ft = ft_src.copy()
bein = bein.loc[bein['Status'] == 'Active']

ft['Created Date'] = pd.to_datetime(ft['Created Date'], dayfirst=True)
ft = ft.loc[ft['Created Date']>= pd.to_datetime("2026-08-26")]
ft = ft.loc[(ft['Doc Type'] == 'Invoice') & (ft['Doc Status']=='Posted')]
ft = ft.loc[ft['Default Entity Type'].isin(['Bein Dealer','Partner Head Office'])]

ft = ft.merge(right=bein[['Customer Number', 'Entity']] , left_on='Subscriber Nr', right_on='Customer Number', how='inner')

ft['Default Entity Type'] = ft['Default Entity Type'].str.lower()
ft['Entity'] = ft['Entity'].str.lower()
ft.loc[ft['Default Entity Type']=='partner head office','Default Entity Type'] = 'bein head office'

ft.loc[ft['Entity'] != ft['Default Entity Type']].to_csv('wrong entity on action.csv', index=False)



different entity same day - History

In [ ]:
history_src.columns

In [ ]:
history = pd.read_parquet("history_JAN_SEP_2026.parquet")
history['Transaction Date'] = pd.to_datetime(history['Transaction Date']).dt.date
history = history.loc[history['Transaction Type'].isin(['Journal Voucher','Create Customer','New Payment'])]
history = history.loc[history['Transaction User']!='Shereen abd allah El-Feky']
# history_jv_payment =  history.loc[history['Transaction Type'].isin(['Journal Voucher','New Payment'])]
# history_jv_payment =  history.loc[history['Transaction Type'].isin(['Journal Voucher','New Payment'])]

grouped_history = history.groupby(['Customer#','Transaction Date']).agg(count =('User Entity','nunique'), depot1 = ('User Entity','min'),depot2 = ('User Entity','max')).reset_index()
grouped_history = grouped_history.loc[grouped_history['depot1']!='beIN Ahmed Orabey']

grouped_history

In [ ]:
grouped_history.loc[grouped_history['count']>1].to_csv('karim case.csv', index=False)

In [ ]:
history.loc[history['Customer#']=='19255135']

In [ ]:
test = pd.read_parquet("history_JAN_SEP_2026.parquet")

In [ ]:
test.loc[test['Customer#']==19168222]

cancel wrong transaction within 3 days violations


In [127]:
history = pd.read_parquet("history_JAN_SEP_2026.parquet")
history_oct = pd.read_csv(r"C:\Users\mturky\Downloads\31070039_ALLUSERHISTDTLS.CSV")
history = pd.concat([history,history_oct])
history['Transaction Date'] = pd.to_datetime(history['Transaction Date'],dayfirst=True).dt.date


In [ ]:

history_2afa = history.copy()


# history_2afa = history_2afa.loc[history_2afa['Transaction Type'].isin(['Journal Voucher','Disconnect Contract','New Payment'])]

history_2afa = history_2afa.loc[history_2afa['Transaction Type'].isin(['Journal Voucher'])]


history_2afa.loc[history_2afa['Description'].str.contains('credit jv',case=False,na=False),'Transaction Type'] = "Credit JV"
history_2afa.loc[history_2afa['Description'].str.contains('debit jv',case=False,na=False),'Transaction Type'] = "Debit JV"

# history_2afa.loc[history_2afa['Transaction Type']=='Disconnect Contract','Transaction Type'] = "Debit JV"
# history_2afa.loc[history_2afa['Transaction Type']=='New Payment','Transaction Type'] = "Credit JV"

history_credit_jv = history_2afa.loc[history_2afa['Transaction Type']=='Credit JV']

history_debit_jv = history_2afa.loc[history_2afa['Transaction Type']=='Debit JV']


# history_debit_jv = history_debit_jv.loc[(history_debit_jv['Description'].str.contains('Cancel wrong transaction approved by admin team',case=False,na=False)) 
#                                         & ((history_debit_jv['Description'].str.contains('ulti',case=False,na=False))
#                                            |(history_debit_jv['Description'].str.contains('prem',case=False,na=False))
#                                            |(history_debit_jv['Description'].str.contains('toget',case=False,na=False)))]

history_credit_jv = history_credit_jv.sort_values(['Customer#','Transaction Date'], ascending=[True,False]).drop_duplicates(subset=['Customer#'],keep='first')
history_debit_jv = history_debit_jv.sort_values(['Customer#','Transaction Date'], ascending=[True,False]).drop_duplicates(subset=['Customer#'],keep='first')

history_credit_jv = history_credit_jv.rename(columns={"Transaction Date":"Credit Transaction Date"})
history_debit_jv = history_debit_jv.rename(columns={"Transaction Date":"Debit Transaction Date"})


merged = pd.merge(left=history_credit_jv[['Customer#','Credit Transaction Date' ]] , right=history_debit_jv[['Customer#','Debit Transaction Date' ]], on="Customer#")
merged['Days Difference'] = (
    merged['Debit Transaction Date'] -
    merged['Credit Transaction Date']
).apply(lambda x: x.days)


merged.loc[merged['Days Difference']>3].to_csv("cancel wrong transaction within 3 days violations.csv", index=False)


Cancel with no debit at same day (not needed)

In [ ]:
history_cancel_no_debit = history.copy()
history_cancel = history_cancel_no_debit.loc[history_cancel_no_debit['Description'].str.contains('Cancel wrong transaction approved by admin team',case=False,na=False)]
history_debit = history_cancel_no_debit.loc[history_cancel_no_debit['Description'].str.contains('debit jv',case=False,na=False)]

history_cancel = (
    history_cancel
    .merge(
        history_debit[['Customer#', 'Transaction Date']].drop_duplicates(),
        on=['Customer#', 'Transaction Date'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
)


history_cancel.to_csv("cancel with no debit same day.csv", index=False)

future dated cancellation (not needed)

In [148]:
history_fut_cancel = history.copy()

history_fut_cancel.loc[(history_fut_cancel['Transaction Type']=='Disconnect Contract') & (history_fut_cancel['Description'].str.contains('Customer cancelled the order successfully',case=False, na=False))].to_csv('fut cancel.csv', index= False)
